In [ ]:
from dotenv import load_dotenv
from agents import set_default_openai_client, Agent, Runner, function_tool
from typing import Dict
from IPython.display import Markdown, display
import os
import requests
import asyncio
from openai import AsyncOpenAI
from pydantic import BaseModel



In [ ]:
load_dotenv(override=True)


In [ ]:
#validate topic

class ResponseFormat(BaseModel):
    is_valid: bool
    """Whether the topic is suitable for academic research """
    reason: str
    """ Reason for the validation """
    topic: str  
    """ Suggested topic """

validate_topic_agent = Agent(
    name="Validate Topic Agent",
    instructions="You are an academic researcher. Given a topic, you validate if this topic is suitable for academic research, if it is not suggest a new topic related to the original topic.",
    output_type=ResponseFormat,
    model="gpt-4o-mini",   
)


validate_topic_tool = validate_topic_agent.as_tool(tool_name="validate_topic", tool_description="Validate the topic of the research paper")


In [ ]:
#Abstract generation

abstract_agent = Agent(
    name="Abstract Generation Agent",
    instructions="You are an academic researcher. Given a topic, you generate a comprehensive abstract for the topic.",
    output_type=str,
    model="gpt-4o-mini",
)

abstract_tool = abstract_agent.as_tool(tool_name="abstract", tool_description="Generate a comprehensive abstract for the topic")


In [ ]:
#introduction generation

introduction_agent = Agent(
    name="Introduction Generation Agent",
    instructions="You are an academic researcher. Given a topic, and a comprehensive abstract, you generate a comprehensive and convincing introduction for the topic.",
    output_type=str,
    model="gpt-4o-mini",
)

introduction_tool = introduction_agent.as_tool(tool_name="introduction", tool_description="Generate a comprehensive and convincing introduction for the topic")


In [ ]:
#Literature Review generation
literature_review_agent = Agent(
    name="Literature Review Generation Agent",
    instructions="You are an academic researcher. Given a topic, a comprehensive introduction, and a comprehensive abstract, you generate a comprehensive literature review for the topic.",
    output_type=str,
    model="gpt-4o-mini",
)

literature_review_tool = literature_review_agent.as_tool(tool_name="literature_review", tool_description="Generate a comprehensive literature review for the topic")

In [ ]:
# Methodology generation
methodology_agent = Agent(
    name="Methodology Generation Agent",
    instructions="You are an academic researcher. Given a topic, a comprehensive introduction, a comprehensive abstract, and a comprehensive literature review, you generate a comprehensive methodology for the topic.",
    output_type=str,
    model="gpt-4o-mini",
)

methodology_tool = methodology_agent.as_tool(tool_name="methodology", tool_description="Generate a comprehensive methodology for the topic")


In [ ]:
#Results / Findings generation
results_agent = Agent(
    name="Results Generation Agent",
    instructions="You are an academic researcher. Given a topic, a comprehensive introduction, a comprehensive abstract, a comprehensive literature review, and a comprehensive methodology, you generate a comprehensive results section for the topic. draw diagrams and tables to illustrate the results where necessary and provide a detailed explanation of the results.",
    output_type=str,
    model="gpt-4o-mini",
)

results_tool = results_agent.as_tool(tool_name="results", tool_description="Generate a comprehensive results section for the topic")


In [ ]:
#Discussion and Conclusion generation
discussion_agent = Agent(
    name="Discussion Generation Agent",
    instructions="You are an academic researcher. Given a topic, a comprehensive introduction, a comprehensive abstract, a comprehensive literature review, a comprehensive methodology, and a comprehensive results section, you generate a comprehensive discussion section for the topic.",
    output_type=str,
    model="gpt-4o-mini",
)

discussion_tool = discussion_agent.as_tool(tool_name="discussion", tool_description="Generate a comprehensive discussion section for the topic")


In [ ]:
#References / Bibliography generation
references_agent = Agent(
    name="References Generation Agent",
    instructions="You are an academic researcher. Given a topic, a comprehensive introduction, a comprehensive abstract, a comprehensive literature review, a comprehensive methodology, a comprehensive results section, and a comprehensive discussion section, you generate a comprehensive references section for the topic.",
    output_type=str,
    model="gpt-4o-mini",
)

references_tool = references_agent.as_tool(tool_name="references", tool_description="Generate a comprehensive references section for the topic")

In [ ]:
#validate the project generated


class ProjectEvaluation(BaseModel):
    well_defined_research_problem: int
    feasibility_and_planning: int
    methodological_rigor: int
    originality_and_contribution: int
    literature_review: int
    clear_structure_and_writing: int

evaluate_project_agent = Agent(
        name="Evaluate Project Agent",
        instructions="""
        You are an academic researcher. Given a topic, a comprehensive introduction, a comprehensive abstract, a comprehensive literature review, a comprehensive methodology, a comprehensive results section, and a comprehensive discussion section, you validate the project generated.
       
       Rate each criteria on a scale of 1 to 5, where 1 is the worst and 5 is the best.
       1. Well-Defined Research Problem: A good project starts with a clear, specific problem that warrants investigation, rather than just a broad topic.
       2. Feasibility and Planning: The project must be realistic, considering time constraints, availability of data, and resources.
       3. Methodological Rigor: Selecting an appropriate methodology—whether qualitative or quantitative—is crucial to produce valid, reliable, and unbiased results.
       4. Originality and Contribution: It should build upon existing scholarly literature to fill a research gap or offer a new interpretation of findings
       5. Literature Review: A comprehensive review demonstrates a thorough understanding of the current state of knowledge
       6. Clear Structure and Writing: The final project should be well-organized, typically including an introduction, literature review, methodology, results, discussion, and conclusion.
""",
        output_type=ProjectEvaluation,

        model="gpt-4o-mini",
    )

    
    
evaluate_project_tool = evaluate_project_agent.as_tool(tool_name="evaluate_project", tool_description="Evaluate the project generated")

In [ ]:
odin_research_agent = Agent(
    name="Odin Research Agent",
    instructions="""
    You are an academic researcher. Given a topic, you generate a comprehensive research paper for the topic.
    if the project is not valid, you should ask the user to provide a new topic.

    if the generated project after evalation is less than 20, you should generate a new project all over again.
    """,
   handoffs=[abstract_agent, introduction_agent, literature_review_agent, methodology_agent, results_agent, discussion_agent, references_agent, evaluate_project_agent],
    output_type=str,
    model="gpt-4o-mini"         )

In [ ]:
from agents import trace


with trace("Generate Research Paper") as proj_trace:
    print(proj_trace.trace_id)
    topic = "AI and its impact on society"
    gen = await Runner.run(odin_research_agent, topic)
    display(Markdown(gen.final_output))
   
    